# AnyTime Universal Benchmark

Run early-exit benchmark for any AnyTime model. Switch by changing **one import line** below.

All HW metrics **per-PID** (process-scoped). Energy = `Î£ (power_w_i Ã— end_to_end_sec_i)` in Joules.

Per-run dir contains:
- `hw_results.json` â€” latency + memory + energy + per-PID GPU/CPU + FLOPs/params
- `quality_results.json` â€” accuracy / F1 / mAP / PPL (only if `skip_quality=False`)

Default sweep = HW-only. Pass `skip_quality=False` to enable quality eval.

Outputs:
- `logs/benchmark/{model}/...` â€” per-run JSONs (nested 2-3 levels deep)
- `results/{model}/` â€” CSV summaries


## 1. Setup

In [1]:
import numpy as np

In [2]:
import sys
import os
from pathlib import Path

REPO_ROOT = Path(".").resolve()
sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

repo root: D:\Research\earlyexit\EarlyExitMonoRepo


In [3]:
# HF login (only needed if pulling from a private repo)
from shared import load_env

load_env()

try:
    from google.colab import userdata

    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN", ""))
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login

    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF: logged in")
else:
    print("HF: no token, public repos only")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF: logged in


## 2. Choose model

**Change ONE import line below** to swap which model gets benchmarked.

In [4]:
# # === SWAP THIS LINE TO PICK MODEL ===
# # from benchmark_config import bert as cfg
# from benchmark_config import llama  as cfg
# # from benchmark_config import vision as cfg
# # from benchmark_config import yolo   as cfg

# print("model :", cfg.NAME)
# print("family:", cfg.MODEL_FAMILY)
# print("out   :", cfg.OUT_DIR)

## 3. Run full sweep

Default = HW-only (`skip_quality=True`). Each run does:
- `profile_hw()` â€” HW + latency + per-PID power/VRAM/CPU + FLOPs/params + EDP
- `evaluate_quality()` â€” (skipped by default)

Override sweep dims via `only_*` kwargs (see `cfg.run_all` signature).


In [5]:
# cfg.run_all(skip_quality=False)          # HW + quality (all datasets)
# # cfg.run_all()                            # HW-only (cnn_dailymail latency only)
# # cfg.run_all(skip_quality=False, only_exit=4)  # single exit smoke test

## 3b. Run ALL models (full sweep)

In [ ]:
# Run ALL models -- HW + quality sweep for every model family.
# Comment out any model you want to skip.
from benchmark_config import bert, llama, vision, yolo

for _cfg in [bert, llama, vision, yolo]:
    print(f"{60*chr(61)}")
    print(f"  {_cfg.NAME} ({_cfg.MODEL_FAMILY})")
    print(60*chr(61))
    try:
        _cfg.run_all(skip_quality=False,dry_run=True)  # HW + quality (all datasets)
    except Exception as _e:
        print(f"[{_cfg.NAME}] run_all failed: {_e}")
        import traceback; traceback.print_exc()


  bert (elasticbert-base)
[bert] DRY RUN: 5 samples per (task, exit) -> D:\Research\earlyexit\EarlyExitMonoRepo\logs.dry_run\benchmark\bert | tasks=['SST-2']
[prepare_data] SST-2: already exists at D:\Research\earlyexit\EarlyExitMonoRepo\AnyTimeBert\glue_data\SST-2, skip


c:\Users\User\.conda\envs\cav\Lib\site-packages\transformers\data\processors\glue.py:285: FutureWarning: This processor will be removed from the library soon, preprocessing should be handled with the Hugging Face Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING.format("processor"), FutureWarning)


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ElasticBertForSequenceClassification LOAD REPORT from: OpenMOSS-Team/elasticbert-base
Key                                              | Status     | 
-------------------------------------------------+------------+-
predictions.transforms.{0...11}.LayerNorm.weight | UNEXPECTED | 
sop_classifier.classifiers.{0...11}.weight       | UNEXPECTED | 
elasticbert.encoder.pooler.{0...10}.dense.bias   | UNEXPECTED | 
sop_classifier.classifiers.{0...11}.bias         | UNEXPECTED | 
predictions.transforms.{0...11}.LayerNorm.bias   | UNEXPECTED | 
elasticbert.encoder.pooler.{0...10}.dense.weight | UNEXPECTED | 
predictions.bias                                 | UNEXPECTED | 
predictions.transforms.{0...11}.dense.bias       | UNEXPECTED | 
predictions.transforms.{0...11}.dense.weight     | UNEXPECTED | 
classifier.bias                                  | MISSING    | 
classifier.weight                                | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from

[bert.benchmark] broadcast trained pooler (slot 11) -> 11 other slots
[bert.benchmark] torch.compile enabled per-layer (12 layers)


...

[evaluate_quality] coco exit=4 sub=P4 mAP50=0.3927 mAP=0.2897 P=0.7101 R=0.3386 F1=0.3418 ECE=0.0487


 24       [15, 12, 9]  1   1501888  models.yolo.DDetect                     [80, [128, 192, 256]]         
 25       [15, 18, 9]  1   1501888  models.yolo.DDetect                     [80, [128, 192, 256]]         
 26      [15, 18, 21]  1   1501888  models.yolo.DDetect                     [80, [128, 192, 256]]         
gelan-s-ee summary: 1132 layers, 13233760 parameters, 13233680 gradients, 55.4 GFLOPs



[yolo.benchmark] broadcast upstream head model.22 -> [23, 24, 25, 26] (params copied=340, shape-skipped=0)
[yolo.benchmark] load (ws=pretrained) missing=370 unexpected=0


Scanning D:\Research\earlyexit\EarlyExitMonoRepo\AnyTimeYolo\datasets\coco\labels\val2017.cache... 4952 images, 48 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 00:00
mAP coco E4/P5:   0%|          | 4/5000 [00:00<05:14, 15.89it/s]

[evaluate_quality] coco exit=4 sub=P5 mAP50=0.2187 mAP=0.1662 P=0.6003 R=0.1979 F1=0.2087 ECE=0.0484


In [7]:
from shared import write_benchmark_csvs
from benchmark_config import bert, llama, vision, yolo
from collections import defaultdict
import pandas as pd

for _cfg in [bert, llama, vision, yolo]:
    out_dir = Path(_cfg.OUT_DIR)
    csv_root = REPO_ROOT / "results" / _cfg.NAME
    csv_root.mkdir(parents=True, exist_ok=True)

    hw_dirs = {p.parent for p in out_dir.rglob("hw_results.json")}
    q_dirs = {p.parent for p in out_dir.rglob("quality_results.json")}
    run_dirs = list(hw_dirs | q_dirs)
    if not run_dirs:
        print(f"[{_cfg.NAME}] no results found, skipping")
        continue
    print(f"[{_cfg.NAME}] found {len(run_dirs)} run dirs ({len(hw_dirs)} hw, {len(q_dirs)} quality)")

    groups = defaultdict(dict)
    for rd in run_dirs:
        rel_parent = rd.parent.relative_to(out_dir).as_posix()
        group_key = rel_parent.replace("/", "_") or "root"
        groups[group_key][rd.name] = rd

    for group_key, runs in sorted(groups.items()):
        csv_dir = csv_root / group_key
        csv_dir.mkdir(parents=True, exist_ok=True)
        write_benchmark_csvs(
            results_files=runs,
            out_dir=csv_dir,
            baseline_key=None,
            method_order=sorted(runs.keys()),
        )
        print(f"  {group_key}: {len(runs)} runs -> {csv_dir}")

print("All CSVs under:", REPO_ROOT / "results")


[bert] no results found, skipping
[llama] no results found, skipping
[vision] no results found, skipping
[yolo] no results found, skipping
All CSVs under: D:\Research\earlyexit\EarlyExitMonoRepo\results


## 4b. Export CSVs — ALL models

## 4. Export CSVs

Recursive walk over per-run dirs (handles 2-3 level nesting per model). Merges
`hw_results.json` + `quality_results.json` if both exist. Groups by parent dir.


In [8]:
# from shared import write_benchmark_csvs
# from collections import defaultdict

# out_dir = Path(cfg.OUT_DIR)
# csv_root = REPO_ROOT / "results" / cfg.NAME
# csv_root.mkdir(parents=True, exist_ok=True)

# hw_dirs = {p.parent for p in out_dir.rglob("hw_results.json")}
# q_dirs = {p.parent for p in out_dir.rglob("quality_results.json")}
# run_dirs = list(hw_dirs | q_dirs)
# print(f"found {len(run_dirs)} run dirs under {out_dir} ({len(hw_dirs)} hw, {len(q_dirs)} quality)")

# groups = defaultdict(dict)
# for rd in run_dirs:
#     rel_parent = rd.parent.relative_to(out_dir).as_posix()
#     group_key = rel_parent.replace("/", "_") or "root"
#     groups[group_key][rd.name] = rd

# for group_key, runs in sorted(groups.items()):
#     csv_dir = csv_root / group_key
#     csv_dir.mkdir(parents=True, exist_ok=True)
#     write_benchmark_csvs(
#         results_files=runs,
#         out_dir=csv_dir,
#         baseline_key=None,
#         method_order=sorted(runs.keys()),
#     )
#     print(f"  wrote {len(runs)} runs -> {csv_dir}")

# print("CSVs under:", csv_root)


## 5. Quick view

In [9]:
import pandas as pd

for csv_name in ['latency.csv', 'energy.csv', 'quality.csv', 'hardware.csv']:
    for f in csv_root.rglob(csv_name):
        rel = f.relative_to(csv_root)
        print(f'=== {rel} ===')
        print(pd.read_csv(f).head(8))
        print()
